# 🔬 Workshop: Cherenkov Events Classifications using a CNN
## Super-Kamiokande × Artificial Intelligence

---

## 🌊 Where are we standing?

In Japan, burried **1,000 meters below the Ikeno Mountain**, lies one of the most extraordinary particle physics detector in the world: **Super-Kamiokande (Super-K)**.

Imagine a giant cylinder of **50,000 tons of ultra-pure water** — so pure that if you were to dip your hand in you would not be able to see your arm froma 30 cm distance. The walls are covered by **13,000 photomultiplier sensors**: enormous glass eyes that detect even an individual photon.

When a subatomic particle penetrates the water **faster than the speed of light in water** (yes, that is possible and does not violate relativity), produces a flash of blue light called **Cherenkov radiation** — the optical equivalent of a sonic *boom* of a supersonic plane. That flash forms a **ring** on the walls of the detector.

> 🔗 **Super-Kamiokande Live Monitor:**  
> **[https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)**  
> Click the link now. You are watching real particles penetrating the detector in this very instant.

---

## ⚛️ ¿What does the Super-Kamiokande look for?

The Great Mistery: **Why does the universe exist?**

The formulae in physics predict that the Big Bang created equal amounts of matter and antimatter. They should have annihilated each other. But here we are. Something broke that perfect equillibrium.

The main culprit candidates are **neutrinos** — massless particles that pierce your body this very insant (and the whole Earth) without actually feeling them. To study their behaviors, the detector needs to distinguis what type of particle produced each event:

| Class | Particle | What does the Cherenkov ring look like? |
|-------|-----------|----------------------------------|
| `mu_like` | Muon (µ) | **Sharp and continuous edge** — travels in a straight line |
| `e_like` | Electron / positron | **Granulated or diffused edge** — creates a particle cascade |

In the past, physics experts went through thousands of events manually. Today, a **neural network** does the same thing in miliseconds and with a precision comparable to that of experts.

---

## 🤖 What are we doing today?

We will train a **Convolutional Neural Network (CNN)** — the same type of AI that your cellphone uses for face recognition or a doctor to detect tumors — to automatically classify Super-K events.

By the end you will have:
- ✅ Loaded and explored real detector images
- ✅ Built and trained your first CNN from zero
- ✅ Evaluated its performance with professional metrics
- ✅ Taken a snapshot of the live monitor and let **your model classify it**

---

## 🗺️ Map of the Workshop

```
Sección 1 → Setup y libraries
Sección 2 → GPU Verification
Sección 3 → Load dataset from Drive
Sección 4 → Random seed + prepare data (70/15/15)
Sección 5 → Visualize mu_like vs e_like images
Sección 6 → Build the CNN layer by layer
Sección 7 → Train the model (20 epochs)
Sección 8 → What does the "network" see? Filters and activations
Sección 9 → Live demo: classify the SK monitor image
Sección 10 → Save the model
```

> 💡 **How to execute:** `Shift + Enter` in each code cell. You do not need to understand every line of code — focus on the concept explained in the text before each cell.

---
## 🛠️ Section 1 — Prepare the environment

### What does this cell do?

Before cooking you must gather the ingredients. This cell imports the **libraries** that we will use:

| Library | Usage |
|----------|----------------|
| **NumPy** | Mathematics with number arrays (matrices, vectors) |
| **Matplotlib** | Graphs training images and curves |
| **scikit-learn** | Model evaluation metrics (accuracy, F1, confusion matrix) |

It also defines auxiliary functions in the case of scikit-learn incompability problem.

> ✅ **Expected result:** `STEP 1 ready. HAS_SKLEARN = True`


In [ ]:
# PASO 1 — Setup without installations or restarts
# Prepare utilities and a pure NumPy fallback in case scikit-learn fails.

import numpy as np
import matplotlib.pyplot as plt

# 1) Safe detection of scikit-learn (without blocking if incompatible binaries are present)
HAS_SKLEARN = True
try:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support
except Exception as e:
    print("Warning: scikit-learn is not usable. I will use NumPy fallback. Detail:", str(e))
    HAS_SKLEARN = False

# 2)Common Helpers (it works both for sklearn and fallback)
def _ensure_label_vectors(y_true, y_pred=None, y_proba=None):
    """
    Convert labels to 1D:
      - y_true: true
      - y_pred: predicted (optional)
      - y_proba: probabilities (optional)
    Returns: (y_true_vec, y_pred_vec)
    """
    y_true = np.asarray(y_true)
    if y_pred is None and y_proba is None:
        raise ValueError("You must provide y_pred or y_proba.")
    if y_pred is None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim == 1:   # binary (prob of positive class)
            y_pred = (y_proba >= 0.5).astype(int)
        else:                    # multiclase
            y_pred = np.argmax(y_proba, axis=1)
    else:
        y_pred = np.asarray(y_pred)
    if y_true.shape[0] != y_pred.shape[0]:
        raise ValueError("y_true and y_pred must have the same length.")
    return y_true, y_pred

# 3) Pure NumPy Fallback for confusion matrix and metrics
def confusion_matrix_numpy(y_true, y_pred, labels=None, normalize=None):
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    idx = {l:i for i,l in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t,p in zip(y_true, y_pred):
        cm[idx[t], idx[p]] += 1
    cmf = cm.astype(float)
    if normalize == "true":
        den = cm.sum(axis=1, keepdims=True).clip(min=1)
        cmf = cm / den
    elif normalize == "pred":
        den = cm.sum(axis=0, keepdims=True).clip(min=1)
        cmf = cm / den
    elif normalize == "all":
        den = cm.sum().clip(min=1)
        cmf = cm / den
    return cm, cmf, labels

def plot_confusion_matrix_numpy(cm_display, labels, title="Confusion Matrix"):
    fig, ax = plt.subplots()
    im = ax.imshow(cm_display, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Prediction")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right"); ax.set_yticklabels(labels)
    for i in range(cm_display.shape[0]):
        for j in range(cm_display.shape[1]):
            v = cm_display[i, j]
            ax.text(j, i, f"{v:.2f}" if cm_display.dtype.kind=='f' else f"{v}",
                    ha="center", va="center")
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

def metrics_numpy(y_true, y_pred):
    acc = (y_true == y_pred).mean()
    labels = np.unique(y_true)
    precs, recs, f1s = [], [], []
    for l in labels:
        tp = ((y_true == l) & (y_pred == l)).sum()
        fp = ((y_true != l) & (y_pred == l)).sum()
        fn = ((y_true == l) & (y_pred != l)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
        precs.append(prec); recs.append(rec); f1s.append(f1)
    return acc, float(np.mean(precs)), float(np.mean(recs)), float(np.mean(f1s))

print("STEP 1 done. HAS_SKLEARN =", HAS_SKLEARN)



In [ ]:
---
## ⚡ Section 2 — Verify GPU

### Why do we need a GPU?

A **CPU** (your laptop's processor) executes instructions one by one, very quickly. A **GPU** (graphic card) executes thousands of operations *at the same time* — exactly what a neural network needs, which implies millions of matrix multiplications in each step.

**This CNN's training:**
- In your CPU → ~30 minutes
- In Colab's GPU T4 → **~1-2 minutes** ✨

**Before executing this cell:**  
`Excution environment` → `Change type of execution environment` → select **GPU (T4)**

> ✅ **Expected result:** At least one GPU-type `PhysicalDevice` must appear on the list.


In [1]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.19.0
GPU disponible: []


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 📦 Section 3 — Set up Google Drive and load dataset

### What is a computer vision dataset?

A **dataset** is a collection of labelled examples — the "textbook" that studies the neural network. Ours contains images in grayscale of Super-K Cherenkov events, organized as follows:

```
SK_mu_e_dataset_v01/
  train/     ← 70% images  →  the network LEARNS here
    mu_like/
    e_like/
  val/       ← 15% images  →  adjust parameters while we train
    mu_like/
    e_like/
  test/      ← 15% images  →  FINAL evaluation (the network never saw them)
    mu_like/
    e_like/
```

**Analogy:** Es como estudiar para un examen con ejercicios de práctica (train), hacer simulacros (val) y finalmente presentar el examen real con preguntas nuevas (test).

**Before running:** Upload `SK_mu_e_dataset_v01.zip` to the `colab_shared`folder in Google Drive.

> ✅ **Expected result:** The 6 folder routes are printed (train/val/test × mu_like/e_like).


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/colab_shared/SK_mu_e_dataset_v01.zip'  # <-- adjust if necessary
DATA_DIR = '/content/dataset'  # destination folder
import os, zipfile, sys

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DATA_DIR)

# Show tree to verify extraction
!find /content/dataset -maxdepth 3 -type d -print


---
## 🎲 Section 4 — Random seed and loading dataset (70 / 15 / 15)

### Why do we fix the seed?

The training of a neutal network uses a lot of random numbers: step initialization, order of images, dropout, etc. Without a fix seed, each execution would give a different result and it would be impossible to reproduce them.

`SEED = 42` ensures that your training is **reproducible** — same seed, same results. It's standard practice in data science and research.

### What does`image_dataset_from_directory` do?

Keras automatically reads the images from the folder structure, it redimensons them to **224×224 pixels**, it converts them to grayscale and it groups them in **lots (batches) of 16** to perform efficiently during the training.

> ✅ **Expected result:** Image counter per split (train / val / test).


In [ ]:
SEED = 42
import os, random, numpy as np, tensorflow as tf
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Automatically detect where train/val/test are located
def find_base_dir(root="/content/dataset"):
    # 1) Typical case: root/dataset/train|val|test
    if all(os.path.isdir(os.path.join(root, "dataset", d)) for d in ["train","val","test"]):
        return os.path.join(root, "dataset")
    # 2) Case: root/train|val|test directly
    if all(os.path.isdir(os.path.join(root, d)) for d in ["train","val","test"]):
        return root
    # 3) Search in subdirectories
    for current, dirs, files in os.walk(root):
        if all(os.path.isdir(os.path.join(current, d)) for d in ["train","val","test"]):
            return current
    raise FileNotFoundError("Cannot find train/val/test directories inside /content/dataset")

BASE_DIR = find_base_dir("/content/dataset")
print("BASE_DIR detected:", BASE_DIR)

train_dir = os.path.join(BASE_DIR, 'train')
val_dir   = os.path.join(BASE_DIR, 'val')
test_dir  = os.path.join(BASE_DIR, 'test')

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='binary',
    color_mode='grayscale', batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, shuffle=True, seed=SEED
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, labels='inferred', label_mode='binary',
    color_mode='grayscale', batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, shuffle=False
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, labels='inferred', label_mode='binary',
    color_mode='grayscale', batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, shuffle=False
)

print("Classes:", train_ds.class_names)

# Normalization and prefetch
norm = tf.keras.layers.Rescaling(1./255)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x,y: (norm(x), y)).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x,y: (norm(x), y)).prefetch(AUTOTUNE)
test_ds  = test_ds.map(lambda x,y: (norm(x), y)).prefetch(AUTOTUNE)


---
## 👁️ Section 5 — Visual exploration of the dataset

### Machine Learning's Golden Rule: always look at your data

Before training any model you should take a look at what is inside the dataset. The questions that a data scientists asks here are:

- Can I easily tell two classes apart?
- How different do `mu_like` and `e_like` look?
- Are the images good quality or is there noise?

### What you will see:

| `mu_like` (muon) | `e_like` (electron/positron) |
|-----------------|-----------------------------|
| Ring with a **sharp and continuous** outline| Ring with a **diffused or granulated** outline |
| The muon travels in a straight line → light concentrated in a clean circle | The electron clashes and bounces off → cascade-dispersed light  |

> 🤔 **Stop and think:** How much time would you take to manually look through 10,000 images? An expert physicists would take a few seconds per image. The neural network takes miliseconds — and with a perfect consistency, without becoming tired.


In [ ]:

import matplotlib.pyplot as plt

def show_batch(dataset, n=6):
    images, labels = next(iter(dataset))
    n = min(n, images.shape[0])
    plt.figure(figsize=(12, 6))
    for i in range(n):
        ax = plt.subplot(2, (n+1)//2, i+1)
        plt.imshow(tf.squeeze(images[i]), cmap='gray')
        lab = 'mu_like' if labels[i].numpy().item() == 1.0 else 'e_like'
        plt.title(lab)
        plt.axis("off")
    plt.show()

show_batch(train_ds, n=6)


---
## 🧠 Section 6 — CNN Architecture

### Why a CNN and not a standard Neural Network?

A standard neural network would look at your image as an enormous list of pixels without any order — it would lose all the information about *what is next to what*. **Convolutional Neural Networks (CNN)** were specifically designed for images because they respect their spacial structure.

### The pieces of our CNN:

#### 🔍 `Conv2D` — Pattern detector
Slides a 3×3 pixel window across the whole image searching for a specific pattern (a border, a curve, a texture). That is a **filter**. The network *automatically* learns which patterns matter in order to distinguish µ from e.

- **Conv2D(32)** — First pass: detects edges and simple textures
- **Conv2D(64)** — Combines edges → detects shapes
- **Conv2D(128)** — Combines shapes → detects complex structures (sharp or diffused ring?)

#### ↓ `MaxPooling2D` — Resolution reduction
After each convolution, the image is reduced by half of the maximum value of each 2x2 square. Result: less parameters and more robustness against small variations.

#### 📊 `Flatten` + `Dense` — Decision-making
The feature maps are flattened into a vector and aconnected to classical neurons that combine all the information to produce a final output.

#### 🎲 `Dropout(0.5)` — "Memorization prevention"
During training, 50% of the neurons are randomly turned off at each step. This forces the network not to rely on a single neuron, preventing **overfitting** (memorizing the data instead of learning general patterns).

#### → `Sigmoid` — Output probability
The last neuron produces a number between 0 and 1:
- Near to **1** → the network predicts `mu_like`
- Near to **0** → the network predicts `e_like`

### Complete diagram:
```
Input: 224 × 224 × 1 (gray image)
        ↓
  Conv2D(32)  + ReLU  →  MaxPool  →  112 × 112 × 32
        ↓
  Conv2D(64)  + ReLU  →  MaxPool  →   56 ×  56 × 64
        ↓
  Conv2D(128) + ReLU  →  MaxPool  →   28 ×  28 × 128
        ↓
  Flatten  →   100,352-value vector
        ↓
  Dense(128) + ReLU  →  Dropout(0.5)
        ↓
  Dense(1) + Sigmoid  →  p(mu_like) ∈ [0, 1]
```

> 🤔 **Stop and think:** `model.summary()` will show the total number of parameters (weights) that the network must learn. How many do you think that will be? Does the magnitude surprise you?

In [ ]:
from tensorflow.keras import layers, models

def build_cnn(input_shape=(224,224,1)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_cnn(input_shape=(224,224,1))
model.summary()

---
## 🏋️ Section 7 — Compiling and training (20 epochs)

### How does a neural network learn?

The training is an automatic cycle of **trial, error, and correction**:

1. The network receives an image and **predicts** whether it is `mu_like` or `e_like`
2. The **error** between the prediction and correct answer is measure using the **loss function* (Binary Crossentropy)
3. The error propagates backwards across the whole network(**backpropagation**)
4. The **Adam optimizer** adjusts each weight in the direction that minimizes the error.
5. It is repeated with the next image → and so on for the entire dataset

An **epoch** = a complete run through all the training images. We run **20 epochs**.

### Glossary of on-screen logs:

| Column | Meaning | What do we want? |
|---------|-------------|----------------|
| `loss` | Error in training | To **minimize it** |
| `accuracy` | Correct % in training | To **maximize it** |
| `val_loss` | Validation error | To also **minimize it** (if it increases `loss` minimized → overfitting) |
| `val_accuracy` | Correctn % in validation  | The **real indicator** of learning |

> ⏱️ With GPU T4 this takes approximately 1-2 minutes.
>
> 👀 Observe how `val_accuracy` changes with each epoch — this is the very moment in which the network improves its comprehension of the Cherenkov rings.


In [ ]:

from tensorflow.keras import optimizers, losses, metrics

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

EPOCHS = 20
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

### 📈 What do these curves mean?

With the dataset that we used (small by design so that the workshop is efficient), the curves can be a little noisy — that is completely normal.

In real experiments like the Super-K, **hundreds of thousands of images** are used and the models have much more sophisticated architectures. However, the principle is exactly the same as the one you have already executed.

**How to detect overfitting?**
- If `accuracy` is 100% but `val_accuracy` estagnates or decreases → the network memorized the training dataset
- It is like the student who memorized the textbook exercises but fails the test that has new questions
- Solutions: more data, more Dropout, regularization, data augmentation

> 🤔 **Reflection:** Does the model always get better with more training epochs? What happens if we train 200 epochs on a small dataset?


---
## 🔬 Section 8 — What does the network "see"? Filters and activations

### Opening the black box

A common critique against neural networks is that they are **black boxes** — they work, but we do not kno whow. In particle physics this matters a lot: the scientists need to understand *why* the model takes each decision before trusting the scientific results.

The **filter visualization and activations** is a **Explicable AI (XAI)** technique that allows us to see inside the network.

### What will you see?

**Filters of the first Conv2D layer:**  
They are the 32 "pattern detectos" that the network learned. A lot of them specialize in edges, brightness changes or specific textures — very similar to how the visual cortex works in mammals.

**Activation maps:**  
They show which regions of an image *activate* each filter. The bright zones idnicate that the filter detected its pattern there. PYou can see how the network "pays attention" to the edge of the Cherenkov ring.

> 🤔 **Observe:** Do the same filters activate with the same intensity in a `mu_like` event and in a `e_like` event? That difference is precisely what allows the network to tell them apart.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Obtain the first Conv2D layer from the model
first_conv = None
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Conv2D):
        first_conv = layer
        break

if first_conv is None:
    raise RuntimeError("No encontré la primera capa Conv2D en el modelo.")

# Obtain a batch from the validation set (if it's empty, try with train_ds)
try:
    images, labels = next(iter(val_ds))
except StopIteration:
    print("val_ds no tiene elementos; usando un batch de train_ds.")
    images, labels = next(iter(train_ds))

# n = number of examples to visualize (up to 4)
n = int(min(4, images.shape[0]))

# Show weights (filters) of the 1st layer
weights, biases = first_conv.get_weights()
w = (weights - weights.min()) / (weights.max() - weights.min() + 1e-9)
num_filters = min(8, w.shape[-1])
plt.figure(figsize=(12, 2))
for i in range(num_filters):
    ax = plt.subplot(1, num_filters, i+1)
    plt.imshow(w[:,:,0,i], cmap='gray')
    plt.axis('off')
plt.suptitle('Filters (1st layer)'); plt.show()

# Activations of the first Conv2D layer for the first n images
act_model = tf.keras.Model(inputs=model.inputs, outputs=first_conv.output)
acts = act_model.predict(images[:n], verbose=0)  # (n,H,W,filters)

plt.figure(figsize=(12, 6))
for i in range(n):  # only first n images
    ax = plt.subplot(2, n, i+1)
    plt.imshow(tf.squeeze(images[i]), cmap='gray')
    plt.title('Input'); plt.axis('off')
    ax = plt.subplot(2, n, i+1+n)
    fmap = acts[i, :, :, 0]
    plt.imshow(fmap, cmap='gray')
    plt.title('f0 Activation'); plt.axis('off')
plt.suptitle('Activations (1st layer)'); plt.show()

---
## 🚀 Section 9 — Real time demo: classify a Super-K event

### Now you are the detector

Your neural network is trained. It is time to try it on a **real** Super-Kamiokande detector event.

### Step-by-step instructions:

**1. Opens the live monitor:**  
👉 [https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)

**2. Identify the "barrel" (central panel):**  
It os the middle horizonal strip — the cylindrical walls of the detector unrolled. The Cherenkov rings are clearest there.

**3. Take a screenshot** and crop it leaving only the barrel panel behind.

**4. Run the cell below** and upload your image when the button appears.

**5. Read the prediction:**
```
Prediction: MU-like | p(mu_like) = 0.91  →  the network is 91% sure that it is a muon.
Prediction: E-like  | p(mu_like) = 0.08  →  the network is 92% sure that it is an electron.
```

> 🤔 **To reflect later:** Was the network right? How will you verify it? What would happen if you uploaded an image that was not from the barrel, for example, an image of your face? What does this say about the limits of an AI model?

> ⚠️ **Note:** Make sure to have executed Section 7 (training) before running this cell, or the model won't be available on memory.

In [ ]:
from google.colab import files
import numpy as np
import matplotlib.pyplot as plt
import os

# 1) Upload file (wait until you confirm in the dialog)
up = files.upload()
if not up:
    raise RuntimeError("No attached file. Try again and select an image.")
FNAME = list(up.keys())[0]
print("Attached file:", FNAME)

# 2) Tolerant loader: try PIL first, and if it fails, use OpenCV
def load_gray_224(path):
    from PIL import Image, UnidentifiedImageError
    try:
        img = Image.open(path).convert('L')   # to grayscale
    except UnidentifiedImageError:
        import cv2
        arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None:
            raise ValueError("Could not read the image (unsupported format). Use PNG or JPG.")
        from PIL import Image
        img = Image.fromarray(arr)
    img = img.resize((224,224))
    arr = np.asarray(img).astype('float32')/255.0
    arr = np.expand_dims(arr, axis=(0, -1))  # (1, 224, 224, 1)
    return arr

# 3) Verifications and warm-up
assert 'model' in globals(), "The model is not in memory. Run the training (Section 7) first."
if not os.path.exists(FNAME):
    raise FileNotFoundError(f"{FNAME} does not exist. Upload the file again.")

_ = model.predict(np.zeros((1,224,224,1), dtype='float32'), verbose=0)  # warm-up (initializes GPU/cuDNN)

# 4) Load the image, predict, and display
x = load_gray_224(FNAME)
prob_mu = float(model.predict(x, verbose=0)[0][0])
pred = "MU-like" if prob_mu > 0.5 else "E-like"
print(f"Prediction: {pred} | p(mu_like)={prob_mu:.2f}")

plt.imshow(x[0,:,:,0], cmap='gray')
plt.title(f"{pred} | p(mu)={prob_mu:.2f}")
plt.axis('off')
plt.show()


---
## 💾 Section 10 — Save the model

### Why save the model?

Training a neural network takes time and computational resources. Once trained, we can **save all the learned weights** in a `.keras` file in order to:

- Use it again without re-training
- Share it with other researchers
- Deploy it in a real-world application
- Continue training where we left-ff (fine-tuning)

At Super-Kamiokande, all the train models are shared between the international collaboration groups — groups in Japan, the U.S.A, Europe and in Mexico work with the same models.

> ✅ **Expected result:** The file `model_mu_e.keras` will show up on your Google Drive.


In [ ]:

SAVE_PATH = '/content/drive/MyDrive/colab_shared/model_mu_e.keras'  # adjust if necessary
model.save(SAVE_PATH)
print("Model saved to:", SAVE_PATH)

---
## What did you learn today?

In this workshop you went trough the complete cycle of a Machine Learning project applied to experimental physics:
| Step | ML Concept | Connection with Super-K |
|------|---------------|---------------------|
| Data loading | Labeled dataset | Real detector images |
| Visual exploration | EDA (Exploratory Data Analysis) | Look at the difference of µ vs e |
| CNN Architecture | Computer vision | Detect patterns in Cherenkov rings |
| Training | Backpropagation + Adam | The network learns to tell particles apart |
| Filters and activations | Explicable aI (XAI) | Understand *why* it classifies that way |
| Live demo | Inference / deployment | Clasify real events from the detector |

### What's next? — Next steps if you wish to go further

- **More data:** getting a bigger dataset drastically improves the performance
- **Data augmentation:** rotate, turn and add noise to the images so the network becomes more robust
- **Transfer learning:** use a pretrained network on millions of images (ResNet, EfficientNet) and adapt it to Super-K
- **Grad-CAM:** XAI advances technique that shows exactly which pixels have the most influence on the final decision
- **Multiclass classification:** add more particles (pions, kaons, atmospheric neutrinos)

---

> *"Particle physics produces more data than any human could ever analyze. Artificial intelligence does not replace the physicist — it liberates him to ask the questions that matter."*
